# Executor completo de validações do monorepo

Este notebook executa as validações automáticas e gera artefatos em `spec/out/`.

In [ ]:
from __future__ import annotations

from pathlib import Path
import subprocess

def find_repo_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for base in candidates:
        if (base / 'package.json').exists() and (base / 'apps').exists():
            return base
    return start

ROOT = find_repo_root(Path.cwd().resolve())
OUT = ROOT / 'spec' / 'out'
OUT.mkdir(parents=True, exist_ok=True)
print('Repo root:', ROOT)
print('Output dir:', OUT)

def run(cmd: str, *, check: bool = True) -> int:
    print(f'\n$ {cmd}')
    p = subprocess.run(cmd, shell=True, cwd=ROOT)
    if check and p.returncode != 0:
        raise RuntimeError(f'Falhou com exit code {p.returncode}: {cmd}')
    return p.returncode

## 1) Verificação de ambiente Python

In [ ]:
run('.venv1/bin/python -m pip --version')
run('.venv1/bin/python -m pip show ipykernel')

## 2) Executar notebook de validação principal

In [ ]:
target_nb = "spec/notebooks_02_validate_current_config_state (1).ipynb"
run('.venv1/bin/python -m pip install -U jupyter nbconvert')
run(f".venv1/bin/jupyter nbconvert --to notebook --execute --output-dir 'spec/out' --output 'notebooks_02_validate_current_config_state.executed.ipynb' '{target_nb}'")
print('✅ Notebook executado em:', OUT / 'notebooks_02_validate_current_config_state.executed.ipynb')

## 3) Validar build/lint/test do monorepo (best effort)

In [ ]:
commands = [
    'npm run -s lint',
    'npm run -s test',
    'npm run -s build',
]

for cmd in commands:
    code = run(cmd, check=False)
    print('-> exit code:', code)

## 4) Exportar resumo rápido de execução

In [ ]:
summary = OUT / 'resumo_execucao_validacoes.txt'
summary.write_text(
    '\n'.join([
        'Validações executadas com notebooks_05_executor_validacoes_completo.ipynb',
        f'Repo root: {ROOT}',
        f'Notebook executado: {OUT / 'notebooks_02_validate_current_config_state.executed.ipynb'}',
        'Comandos: npm run -s lint | npm run -s test | npm run -s build',
    ]),
    encoding='utf-8'
)
print('✅ Resumo salvo em:', summary)